<a href="https://colab.research.google.com/github/johanhoffman/DD2365_FEniCSx/blob/main/Shallow-Water-Equations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# About the code

In [ ]:
# Copyright (C) 2019-2026 Johan Hoffman (jhoffman@kth.se)
#
# This file is part of the course DD2365 Advanced Computation in Fluid Mechanics
# KTH Royal Institute of Technology, Stockholm, Sweden
#
# This is free software: you can redistribute it and/or modify
# it under the terms of the GNU Lesser General Public License as published by
# the Free Software Foundation, either version 3 of the License, or
# (at your option) any later version.
#
# This template is maintained by Johan Hoffman
# Please report problems to jhoffman@kth.se

# Setup environment

In [ ]:
# --- canonical: bootstrap v1 ---
import sys, os, subprocess

_on_colab = "google.colab" in sys.modules

if not _on_colab:
    os.environ.setdefault("OMP_NUM_THREADS", "1")

if _on_colab:
    try:
        import gmsh
    except ImportError:
        subprocess.run(
            'wget -q "https://fem-on-colab.github.io/releases/gmsh-install.sh"'
            ' -O /tmp/gmsh-install.sh && bash /tmp/gmsh-install.sh',
            shell=True, check=True,
        )
    try:
        import dolfinx
    except ImportError:
        subprocess.run(
            'wget -q "https://fem-on-colab.github.io/releases/fenicsx-install-release-real.sh"'
            ' -O /tmp/fenicsx-install.sh && bash /tmp/fenicsx-install.sh',
            shell=True, check=True,
        )

import dolfinx
print("dolfinx version:", dolfinx.__version__)

if _on_colab:
    from google.colab import files
# --- end canonical: bootstrap ---

In [ ]:
# --- canonical: gmsh_rect_minus_circles v6 ---
import math
import numpy as np
import gmsh
from mpi4py import MPI
from dolfinx.io import gmsh as gmshio


_ALPHA = 0.66
# mshr numSamples=100 → N = floor(sqrt(100.5)) = 10 interior sample rows
_N_SAMPLE = int(math.floor(math.sqrt(100.0 + 0.5)))  # 10


def _mshr_segments(r, L, H, resolution):
    """Exact mshr auto-segment count for Circle(segments=0) in 2D.

    Source path:
      MeshGenerator.cpp: segment_granularity = 2*R_enc/resolution
      CSGGeometry.cpp:   estimate_bounding_sphere samples _N_SAMPLE×_N_SAMPLE
                         interior points; extreme corners dominate, giving
                         R_enc = ((N-1)/N) * sqrt(L²+H²) / 2
      CSGCGALDomain2D.cpp: num_segs = max(5, round(2π*r / segment_granularity))

    Verified predictions (no Steiner points since edge_len < cs_mshr):
      4×2 domain, r=0.2, resolution=32 → 10
      4×4 domain, r=0.2, resolution=32 →  8
    """
    R_enc = ((_N_SAMPLE - 1) / _N_SAMPLE) * math.sqrt(L ** 2 + H ** 2) / 2.0
    cs = 2.0 * R_enc / resolution
    return max(5, round(2.0 * math.pi * r / cs))


def gmsh_rect_minus_circles(L, H, circles, resolution, segments=32,
                             dist_min=None, dist_max=None):
    """Rectangle [0,L]×[0,H] minus regular-polygon holes.

    Parameters
    ----------
    L, H : float
        Rectangle dimensions.
    circles : list of (cx, cy, r)
        Circle centres and radii.
    resolution : int
        Mesh density parameter.  Interior mesh size:
            lc = _ALPHA * sqrt(L² + H²) / resolution
    segments : int or "mshr"
        Number of polygon sides used to approximate each circle hole.
        Default 32.  Pass ``"mshr"`` to use the legacy mshr auto-segment
        rule (max(5, round(2π·r/cs)), cs derived from bounding-sphere
        estimate), which gives 10 sides for 4×2 domains and 8 for 4×4.
    dist_min : float or None
        Threshold DistMin per hole (inner boundary of transition zone).
        Default None → 0.0 (fine zone starts at the polygon surface).
    dist_max : float or None
        Threshold DistMax per hole (outer boundary of transition zone).
        Default None → r/2 per hole (reproduces v5 behaviour exactly).

    Returns
    -------
    msh : dolfinx.mesh.Mesh
    cell_tags : dolfinx.mesh.MeshTags   (fluid domain tag = 10)

    Notes
    -----
    Vertex placement: phi_i = 2π·i/n, i = 0 … n-1 (i=0 at angle 0, CCW).

    Mesh sizing: one Distance+Threshold gmsh field per polygon transitions
    from ``edge_len = 2r·sin(π/n)`` right at the polygon boundary (prevents
    Steiner-point insertion on polygon edges) to ``lc`` at distance
    ``dist_max`` (default r/2).  No MeshSizeMax override; background field
    is the sole size control.
    """
    lc = _ALPHA * math.sqrt(L ** 2 + H ** 2) / resolution
    eps = 1e-6 * max(L, H)

    gmsh.initialize()
    gmsh.option.setNumber("General.Terminal", 0)

    gmsh.model.occ.addRectangle(0, 0, 0, L, H, tag=1)

    hole_tags = []
    segs_per_circle = []
    for cx, cy, r in circles:
        if segments == "mshr":
            n = _mshr_segments(r, L, H, resolution)
        elif isinstance(segments, int):
            n = segments
        else:
            raise ValueError(f"segments must be an int or 'mshr', got {segments!r}")
        segs_per_circle.append(n)
        pts = [
            gmsh.model.occ.addPoint(
                cx + r * math.cos(2 * math.pi * i / n),
                cy + r * math.sin(2 * math.pi * i / n),
                0,
            )
            for i in range(n)
        ]
        lines = [
            gmsh.model.occ.addLine(pts[i], pts[(i + 1) % n])
            for i in range(n)
        ]
        cl = gmsh.model.occ.addCurveLoop(lines)
        surf = gmsh.model.occ.addPlaneSurface([cl])
        hole_tags.append((2, surf))

    if hole_tags:
        gmsh.model.occ.cut([(2, 1)], hole_tags)
    gmsh.model.occ.synchronize()

    # Distance+Threshold field per polygon hole
    bg_fields = []
    for i, (cx, cy, r) in enumerate(circles):
        n = segs_per_circle[i]
        edge_len = 2 * r * math.sin(math.pi / n)

        # Identify polygon curves (not on the rectangle boundary)
        c_curves = []
        for dim, tag in gmsh.model.getEntities(1):
            xmin, ymin, _, xmax, ymax, _ = gmsh.model.getBoundingBox(dim, tag)
            if xmin <= eps or xmax >= L - eps or ymin <= eps or ymax >= H - eps:
                continue
            bounds = gmsh.model.getParametrizationBounds(dim, tag)
            mid = float(np.array(bounds).mean())
            x, y, _ = gmsh.model.getValue(dim, tag, [mid])
            if abs(math.hypot(x - cx, y - cy) - r) < 0.15 * r:
                c_curves.append(tag)

        if not c_curves:
            continue

        dm_min = 0.0 if dist_min is None else dist_min
        dm_max = r / 2.0 if dist_max is None else dist_max

        d_id = 10 + 2 * i
        t_id = 11 + 2 * i
        gmsh.model.mesh.field.add("Distance", d_id)
        gmsh.model.mesh.field.setNumbers(d_id, "CurvesList", c_curves)
        gmsh.model.mesh.field.add("Threshold", t_id)
        gmsh.model.mesh.field.setNumber(t_id, "InField", d_id)
        # SizeMin = edge_len at polygon: prevents subdivision of polygon edges.
        # SizeMax = lc at dist_max: interior mesh size.
        gmsh.model.mesh.field.setNumber(t_id, "SizeMin", edge_len)
        gmsh.model.mesh.field.setNumber(t_id, "SizeMax", lc)
        gmsh.model.mesh.field.setNumber(t_id, "DistMin", dm_min)
        gmsh.model.mesh.field.setNumber(t_id, "DistMax", dm_max)
        bg_fields.append(t_id)

    if bg_fields:
        if len(bg_fields) == 1:
            gmsh.model.mesh.field.setAsBackgroundMesh(bg_fields[0])
        else:
            min_id = 10 + 2 * len(circles)
            gmsh.model.mesh.field.add("Min", min_id)
            gmsh.model.mesh.field.setNumbers(min_id, "FieldsList", bg_fields)
            gmsh.model.mesh.field.setAsBackgroundMesh(min_id)

    # Background field is the sole size control
    gmsh.option.setNumber("Mesh.MeshSizeExtendFromBoundary", 0)
    gmsh.option.setNumber("Mesh.MeshSizeFromPoints", 0)
    gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 0)
    gmsh.option.setNumber("Mesh.Algorithm", 5)  # Delaunay

    surfaces = gmsh.model.getEntities(2)
    gmsh.model.addPhysicalGroup(2, [s[1] for s in surfaces], tag=10, name="domain")
    gmsh.model.mesh.generate(2)

    mesh_data = gmshio.model_to_mesh(gmsh.model, MPI.COMM_WORLD, rank=0, gdim=2)
    gmsh.finalize()
    return mesh_data.mesh, mesh_data.cell_tags
# --- end canonical: gmsh_rect_minus_circles ---

In [ ]:
# --- canonical: refine_cells v1 ---
import numpy as np
import dolfinx.mesh
from dolfinx.mesh import RefinementOption


def refine_cells(msh, predicate_or_mask):
    """Refine selected cells using the Plaza algorithm.

    Parameters
    ----------
    msh : dolfinx.mesh.Mesh
    predicate_or_mask : callable or array-like of bool
        Either a callable ``f(midpoints) -> bool array`` where
        ``midpoints`` has shape ``(ncells, gdim)``, or a boolean array
        of length ``num_local_cells`` that directly marks which cells
        to refine (useful when marks come from an existing DG0 field).

    Returns
    -------
    refined_msh : dolfinx.mesh.Mesh
    parent_cells : np.ndarray[np.int32]  shape (num_refined_cells,)
    parent_facets : np.ndarray[np.int8]  shape (num_refined_cells,)
        Local parent-facet index per refined cell, or -1 for interior.
    """
    tdim = msh.topology.dim
    num_cells = msh.topology.index_map(tdim).size_local

    if callable(predicate_or_mask):
        msh.topology.create_entities(1)
        msh.topology.create_connectivity(tdim, 0)
        midpoints = dolfinx.mesh.compute_midpoints(
            msh, tdim, np.arange(num_cells, dtype=np.int32)
        )
        marked = np.where(predicate_or_mask(midpoints))[0].astype(np.int32)
    else:
        mask = np.asarray(predicate_or_mask, dtype=bool)
        marked = np.where(mask[:num_cells])[0].astype(np.int32)

    if len(marked) == 0:
        n = msh.topology.index_map(tdim).size_local
        return msh, np.arange(n, dtype=np.int32), np.full(n, -1, dtype=np.int8)

    msh.topology.create_entities(1)
    msh.topology.create_connectivity(tdim, 1)
    edges = dolfinx.mesh.compute_incident_entities(msh.topology, marked, tdim, 1)
    edges = np.unique(edges).astype(np.int32)
    return dolfinx.mesh.refine(
        msh, edges, option=RefinementOption.parent_cell_and_facet
    )
# --- end canonical: refine_cells ---

In [ ]:
# --- canonical: tag_boundaries v1 ---
import numpy as np
from dolfinx.mesh import locate_entities_boundary, meshtags


def tag_boundaries(msh, L, H, eps=None):
    """Tag exterior boundary facets of a [0,L]×[0,H] rectangle with holes.

    Assigns integer tags to all exterior boundary facets:

        left=1  (x ≈ 0),   right=2 (x ≈ L),
        lower=3 (y ≈ 0),   upper=4 (y ≈ H),
        objects=5 (remaining exterior facets — circle boundaries).

    Corners are unambiguous: boundary facets are edges, not vertices, so a
    corner vertex is shared by one vertical and one horizontal edge — each
    edge belongs to exactly one group and no facet is tagged twice.

    Parameters
    ----------
    msh : dolfinx.mesh.Mesh
    L, H : float
        Rectangle dimensions.
    eps : float, optional
        Coordinate tolerance.  Default ``1e-6 * max(L, H)``.

    Returns
    -------
    dolfinx.mesh.MeshTags
        Must be recomputed whenever the mesh changes (e.g. after refinement).
    """
    if eps is None:
        eps = 1e-6 * max(L, H)

    fdim = msh.topology.dim - 1
    msh.topology.create_entities(fdim)
    msh.topology.create_connectivity(fdim, msh.topology.dim)

    left  = locate_entities_boundary(msh, fdim, lambda x: x[0] <= eps)
    right = locate_entities_boundary(msh, fdim, lambda x: x[0] >= L - eps)
    lower = locate_entities_boundary(msh, fdim, lambda x: x[1] <= eps)
    upper = locate_entities_boundary(msh, fdim, lambda x: x[1] >= H - eps)

    known = np.unique(np.concatenate([left, right, lower, upper]))
    all_bdry = locate_entities_boundary(
        msh, fdim, lambda x: np.ones(x.shape[1], dtype=bool)
    )
    objects = np.setdiff1d(all_bdry, known)

    indices = np.concatenate([left, right, lower, upper, objects]).astype(np.int32)
    values  = np.concatenate([
        np.full(len(left),    1, dtype=np.int32),
        np.full(len(right),   2, dtype=np.int32),
        np.full(len(lower),   3, dtype=np.int32),
        np.full(len(upper),   4, dtype=np.int32),
        np.full(len(objects), 5, dtype=np.int32),
    ])
    order = np.argsort(indices)
    return meshtags(msh, fdim, indices[order], values[order])
# --- end canonical: tag_boundaries ---

In [ ]:
# --- canonical: plot_helpers v3 ---
import numpy as np
import matplotlib.pyplot as plt
import basix.ufl as _bufl
from dolfinx.fem import functionspace, Function


def plot_mesh(msh, title="Mesh"):
    """Plot a 2-D triangular mesh using matplotlib triplot."""
    msh.topology.create_connectivity(msh.topology.dim, 0)
    x = msh.geometry.x
    gdm = msh.geometry.dofmaps[0]
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.triplot(x[:, 0], x[:, 1], gdm, linewidth=0.3, color="k")
    ax.set_aspect("equal")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def _p1_scalar_space(msh):
    return functionspace(msh, ("Lagrange", 1))


def _scatter_p1_scalar(f1):
    """Scatter a P1 scalar Function values to geometry nodes.

    Returns (x, geometry_connectivity, node_values).
    """
    V = f1.function_space
    msh = V.mesh
    x = msh.geometry.x
    gdm = msh.geometry.dofmaps[0]
    ldm = V.dofmap.list
    vals = np.zeros(x.shape[0])
    vals[gdm.ravel()] = f1.x.array.real[ldm.ravel()]
    return x, gdm, vals


def plot_p1(u, title="Solution"):
    """Plot a P1 scalar Function using matplotlib tripcolor (Gouraud shading)."""
    plot_scalar(u, title=title)


def plot_scalar(f, title="Scalar field", ax=None):
    """Plot any scalar Lagrange Function via tripcolor (interpolates to P1 if needed).

    Parameters
    ----------
    f : dolfinx.fem.Function   scalar; any Lagrange degree
    title : str
    ax : matplotlib.axes.Axes or None
        If given, draw into this axes (no new figure, no plt.show()).
        If None (default), create a new figure and call plt.show().
    """
    V = f.function_space
    msh = V.mesh
    el = V.ufl_element()
    if el.degree == 1 and el.reference_value_shape == ():
        f1 = f
    else:
        f1 = Function(_p1_scalar_space(msh))
        f1.interpolate(f)
    x, gdm, vals = _scatter_p1_scalar(f1)
    _standalone = ax is None
    if _standalone:
        fig, ax = plt.subplots(figsize=(8, 3))
    tc = ax.tripcolor(x[:, 0], x[:, 1], gdm, vals, shading="gouraud")
    plt.colorbar(tc, ax=ax)
    ax.set_aspect("equal")
    ax.set_title(title)
    if _standalone:
        plt.tight_layout()
        plt.show()


def plot_vector(u, title="Vector field", quiver=True, quiver_stride=8, ax=None):
    """Plot a 2-D vector Lagrange Function as colour map of |u| + optional quiver.

    Parameters
    ----------
    u : dolfinx.fem.Function   value shape (2,); any Lagrange degree
    title : str
    quiver : bool              overlay subsampled arrows (default True)
    quiver_stride : int        take every N-th mesh node for arrows
    ax : matplotlib.axes.Axes or None
        If given, draw into this axes (no new figure, no plt.show()).
        If None (default), create a new figure and call plt.show().
    """
    msh = u.function_space.mesh
    gdim = msh.geometry.dim
    x = msh.geometry.x
    gdm = msh.geometry.dofmaps[0]
    npts = x.shape[0]

    # Interpolate to P1 vector so values align with geometry nodes
    V1v = functionspace(msh, _bufl.element("Lagrange", msh.basix_cell(), 1, shape=(gdim,)))
    u1v = Function(V1v)
    u1v.interpolate(u)

    # Scatter: for a block-size-2 P1 space, dofmap.list gives block indices
    ldm = V1v.dofmap.list      # (ncells, 3)  — block indices
    arr = u1v.x.array.real     # length npts * gdim, interleaved per block
    ux = np.zeros(npts)
    uy = np.zeros(npts)
    ux[gdm.ravel()] = arr[ldm.ravel() * gdim + 0]
    uy[gdm.ravel()] = arr[ldm.ravel() * gdim + 1]
    mag = np.sqrt(ux**2 + uy**2)

    _standalone = ax is None
    if _standalone:
        fig, ax = plt.subplots(figsize=(8, 3))
    tc = ax.tripcolor(x[:, 0], x[:, 1], gdm, mag, shading="gouraud", cmap="viridis")
    plt.colorbar(tc, ax=ax, label="|u|")
    if quiver:
        idx = np.arange(0, npts, quiver_stride)
        ax.quiver(x[idx, 0], x[idx, 1], ux[idx], uy[idx],
                  color="white", alpha=0.6, width=0.002, scale_units="xy", scale=2.0)
    ax.set_aspect("equal")
    ax.set_title(title)
    if _standalone:
        plt.tight_layout()
        plt.show()
# --- end canonical: plot_helpers ---

In [ ]:
# --- canonical: xdmf_series v1 ---
import sys
import pathlib
import basix.ufl as _bufl
from mpi4py import MPI
from dolfinx.io import XDMFFile
from dolfinx.fem import functionspace, Function


def _to_p1(f):
    """Interpolate f to P1 (scalar or vector) if needed."""
    V = f.function_space
    msh = V.mesh
    el = V.ufl_element()
    vshape = el.reference_value_shape
    if el.degree == 1:
        return f
    if vshape == ():
        V1 = functionspace(msh, ("Lagrange", 1))
    else:
        V1 = functionspace(msh, _bufl.element("Lagrange", msh.basix_cell(), 1, shape=vshape))
    f1 = Function(V1, name=f.name)
    f1.interpolate(f)
    return f1


class XDMFSeries:
    """Time-series XDMF writer.

    Opens one XDMFFile, writes the mesh once on the first write() call,
    then appends each time step.  Higher-degree functions are automatically
    interpolated to P1 before writing.

    Usage::

        xdmf = XDMFSeries("output/solution.xdmf")
        for t in ...:
            xdmf.write([u1, p1, w1], t)
        xdmf.close()          # or: `with XDMFSeries(...) as xdmf:`

    Parameters
    ----------
    path : str or Path
        Output ``.xdmf`` file path.  The ``.h5`` sidecar is written alongside.
    download : bool, optional
        If True *and* running on Colab, tar the pair and trigger a download
        when :meth:`close` is called.
    """

    def __init__(self, path, download=False):
        self._path = pathlib.Path(path)
        self._path.parent.mkdir(parents=True, exist_ok=True)
        self._download = download
        self._xdmf = XDMFFile(MPI.COMM_WORLD, str(self._path), "w")
        self._mesh_written = False

    def write(self, funcs, t):
        """Write *funcs* at time *t*."""
        funcs_p1 = [_to_p1(f) for f in funcs]
        if not self._mesh_written:
            self._xdmf.write_mesh(funcs_p1[0].function_space.mesh)
            self._mesh_written = True
        for f in funcs_p1:
            self._xdmf.write_function(f, t)

    def close(self):
        """Close the file and optionally trigger a Colab download."""
        self._xdmf.close()
        if self._download and "google.colab" in sys.modules:
            import tarfile
            from google.colab import files as colab_files
            tar_path = str(self._path.with_suffix(".tar.gz"))
            with tarfile.open(tar_path, "w:gz") as tar:
                tar.add(str(self._path), arcname=self._path.name)
                h5 = self._path.with_suffix(".h5")
                if h5.exists():
                    tar.add(str(h5), arcname=h5.name)
            colab_files.download(tar_path)

    def __enter__(self):
        return self

    def __exit__(self, *_):
        self.close()
# --- end canonical: xdmf_series ---

In [ ]:
# --- canonical: time_step v1 ---
import numpy as np


def time_step(msh, U_ref, C_CFL=0.5):
    """Return C_CFL * h_min / U_ref (CFL-based time step)."""
    tdim = msh.topology.dim
    n_cells = msh.topology.index_map(tdim).size_local
    h_min = float(msh.h(tdim, np.arange(n_cells, dtype=np.int32)).min())
    return C_CFL * h_min / U_ref
# --- end canonical: time_step ---

In [ ]:
import numpy as np
import time
import math
from mpi4py import MPI
from dolfinx.fem import (
    functionspace, Function, Constant, form,
    assemble_scalar, locate_dofs_topological, dirichletbc,
)
from dolfinx.fem.petsc import (
    assemble_matrix, assemble_vector, apply_lifting, set_bc,
    create_matrix, create_vector,
)
from petsc4py import PETSc
import ufl
import basix.ufl as bufl
import matplotlib.pyplot as plt

# **Introduction**

The Shallow Water equations model depth-averaged free-surface flows. The velocity field $u$ and wave height $w$ satisfy

$$\dot u + (u\cdot\nabla)u + g\nabla w = 0$$

$$\dot w + \nabla\cdot(wu) = 0$$

We use a GLS-stabilized P1/P1 fractional-step scheme with SUPG stabilization on both the momentum and continuity equations.

# **Method**

**Define domain and mesh**

In [ ]:
# Define rectangular domain
L = 4
H = 2

# Define circle
xc = 1.0
yc = 0.5 * H
rc = 0.2

# Generate mesh
resolution = 32
msh, _ = gmsh_rect_minus_circles(L, H, [(xc, yc, rc)], resolution)
facet_tags = tag_boundaries(msh, L, H)

# Local mesh refinement (no_levels=0 → no refinement)
no_levels = 0
for _ in range(no_levels):
    msh, _, _ = refine_cells(
        msh,
        lambda midpoints: np.hypot(midpoints[:, 0] - xc, midpoints[:, 1] - yc) < 1.0,
    )
    facet_tags = tag_boundaries(msh, L, H)

tdim = msh.topology.dim
fdim = tdim - 1
n_cells = msh.topology.index_map(tdim).size_local

plot_mesh(msh)

**Define finite element approximation spaces**

In [ ]:
# Generate finite element spaces (for velocity and wave height)
P1v = bufl.element("Lagrange", msh.basix_cell(), 1, shape=(2,))
P1s = bufl.element("Lagrange", msh.basix_cell(), 1)

V = functionspace(msh, P1v)   # velocity
Q = functionspace(msh, P1s)   # wave height

# Define trial and test functions
u = ufl.TrialFunction(V)
w = ufl.TrialFunction(Q)
v = ufl.TestFunction(V)
q = ufl.TestFunction(Q)

**Define boundary conditions**

In [ ]:
# Define boundary conditions
def _bc(space, value, tag):
    dofs = locate_dofs_topological(space, fdim, facet_tags.find(tag))
    return dirichletbc(PETSc.ScalarType(value), dofs, space)

# Velocity BCs (exactly as legacy)
bcu_in0  = _bc(V.sub(0), 1.0, 1)   # u_x = 1  at inflow (left)
bcu_in1  = _bc(V.sub(1), 0.0, 1)   # u_y = 0  at inflow
bcu_upp1 = _bc(V.sub(1), 0.0, 4)   # u_y = 0  at upper wall (slip: u_x free)
bcu_low1 = _bc(V.sub(1), 0.0, 3)   # u_y = 0  at lower wall (slip: u_x free)
bcu_obj0 = _bc(V.sub(0), 0.0, 5)   # u_x = 0  on cylinder (no-slip)
bcu_obj1 = _bc(V.sub(1), 0.0, 5)   # u_y = 0  on cylinder

bcu = [bcu_in0, bcu_in1, bcu_upp1, bcu_low1, bcu_obj0, bcu_obj1]

# Wave-height BC: w = |sin(t)| at inflow (time-dependent; updated each step)
w_in = Constant(msh, PETSc.ScalarType(0.0))
dofs_w_in = locate_dofs_topological(Q, fdim, facet_tags.find(1))
bcw = [dirichletbc(w_in, dofs_w_in, Q)]

ds_m = ufl.Measure("ds", domain=msh, subdomain_data=facet_tags)

# **Results**

**Define flow parameters**

In [ ]:
# Flow parameters
g = 1.0

# Simulation parameters
T = 30.0
plot_freq = 10

**Define method parameters**

In [ ]:
# Solution functions
u0 = Function(V, name="u0"); u1 = Function(V, name="u1")
w0 = Function(Q, name="w0"); w1 = Function(Q, name="w1")

num_nnlin_iter = 5

# Time step: CFL-based, dt = 0.5 * hmin
dt = time_step(msh, 1.0)
dt_c = Constant(msh, PETSc.ScalarType(dt))
g_c  = Constant(msh, PETSc.ScalarType(g))

# Force computation setup (lift: phi_y=1; sampled once per step after nonlinear loop)
# Use both components so phi=(1,0) gives drag, phi=(0,1) gives lift
phi_x = 0.0
phi_y = 1.0
psi_u = Function(V)
V0c, map_to_V_0 = V.sub(0).collapse()
V1c, map_to_V_1 = V.sub(1).collapse()
psi_0c = Function(V0c)
psi_1c = Function(V1c)
cyl_dofs_0c = locate_dofs_topological(V0c, fdim, facet_tags.find(5))
cyl_dofs_1c = locate_dofs_topological(V1c, fdim, facet_tags.find(5))
psi_0c.x.array[cyl_dofs_0c] = phi_x
psi_1c.x.array[cyl_dofs_1c] = phi_y
psi_u.x.array[map_to_V_0] = psi_0c.x.array
psi_u.x.array[map_to_V_1] = psi_1c.x.array
psi_u.x.scatter_forward()
D = 2 * rc
normalization = -2.0 / D
force_array = []
time_array  = []
start_sample_time = 1.0

**Define variational problem**

In [ ]:
# Stabilization parameters
h_c   = ufl.CellDiameter(msh)
u_mag = ufl.sqrt(ufl.dot(u1, u1))
d1    = 2.0 / ufl.sqrt((1.0 / dt_c)**2 + (u_mag / h_c)**2)
d2    = h_c * u_mag   # defined for legacy fidelity; not used in forms

# Trapezoidal averages
um  = 0.5 * (u  + u0)    # involves TrialFunction u (for lhs/rhs split)
um1 = 0.5 * (u1 + u0)    # known (frozen advection velocity)
wm1 = 0.5 * (w1 + w0)    # known wave height average

# Momentum variational form (velocity)
Fu = (ufl.inner((u - u0)/dt_c + ufl.dot(ufl.grad(um), um1), v)*ufl.dx
    + g_c * ufl.inner(ufl.grad(wm1), v)*ufl.dx
    + d1 * ufl.inner(ufl.dot(ufl.grad(um), um1), ufl.dot(ufl.grad(v), um1))*ufl.dx)

# Continuity variational form (wave height)
Fw = (ufl.inner((w - w0)/dt_c, q)*ufl.dx
    + ufl.div(w * um1)*q*ufl.dx
    + d1 * ufl.inner(ufl.dot(um1, ufl.grad(w)), ufl.dot(um1, ufl.grad(q)))*ufl.dx)

a_u = form(ufl.lhs(Fu)); L_u = form(ufl.rhs(Fu))
a_w = form(ufl.lhs(Fw)); L_w = form(ufl.rhs(Fw))

# Force residual (volume form; +g corrected from legacy -g)
Force_form = form(
    ufl.inner((u1 - u0)/dt_c + ufl.dot(ufl.grad(um1), um1), psi_u)*ufl.dx
  + g_c * ufl.inner(ufl.grad(wm1), psi_u)*ufl.dx
)

In [ ]:
A_u = create_matrix(a_u); b_u = create_vector(V)
A_w = create_matrix(a_w); b_w = create_vector(Q)

def _ksp_ilu(A):
    k = PETSc.KSP().create(msh.comm); k.setOperators(A); k.setType("bcgs")
    k.getPC().setType("ilu"); k.setTolerances(rtol=1e-6, atol=1e-14, max_it=500)
    k.setFromOptions(); return k

def _ksp_amg(A):
    k = PETSc.KSP().create(msh.comm); k.setOperators(A); k.setType("bcgs")
    pc = k.getPC(); pc.setType("hypre"); pc.setHYPREType("boomeramg")
    k.setTolerances(rtol=1e-6, atol=1e-14, max_it=500); k.setFromOptions(); return k

ksp_u = _ksp_ilu(A_u)
ksp_w = _ksp_amg(A_w)

# Apply BCs to initial conditions
set_bc(u0.x.petsc_vec, bcu); u0.x.scatter_forward()
set_bc(u1.x.petsc_vec, bcu); u1.x.scatter_forward()

**Set plotting variables and open export files**

In [ ]:
xdmf = XDMFSeries("results-SWE/solution.xdmf")

plot_time = 0.0

**Time stepping algorithm**

In [ ]:
t_wall0 = time.time()
t = dt
step = 0
while t < T + 1e-10:

    # Update time-dependent wave-height inflow BC: w_in = |sin(t)|
    w_in.value = abs(math.sin(t))

    # Solve non-linear problem
    k = 0
    while k < num_nnlin_iter:

        # Assemble and solve momentum (velocity)
        A_u.zeroEntries()
        assemble_matrix(A_u, a_u, bcs=bcu); A_u.assemble()
        with b_u.localForm() as loc: loc.set(0.0)
        assemble_vector(b_u, L_u)
        apply_lifting(b_u, [a_u], bcs=[bcu])
        b_u.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
        set_bc(b_u, bcu)
        ksp_u.setOperators(A_u, A_u); ksp_u.solve(b_u, u1.x.petsc_vec)
        u1.x.scatter_forward()
        set_bc(u1.x.petsc_vec, bcu); u1.x.scatter_forward()

        # Assemble and solve continuity (wave height)
        A_w.zeroEntries()
        assemble_matrix(A_w, a_w, bcs=bcw); A_w.assemble()
        with b_w.localForm() as loc: loc.set(0.0)
        assemble_vector(b_w, L_w)
        apply_lifting(b_w, [a_w], bcs=[bcw])
        b_w.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
        set_bc(b_w, bcw)
        ksp_w.setOperators(A_w, A_w); ksp_w.solve(b_w, w1.x.petsc_vec)
        w1.x.scatter_forward()
        set_bc(w1.x.petsc_vec, bcw); w1.x.scatter_forward()

        k += 1

    # Force (sampled once per step after nonlinear convergence)
    F_val = float(msh.comm.allreduce(assemble_scalar(Force_form), op=MPI.SUM))
    if t > start_sample_time:
        force_array.append(normalization * F_val)
        time_array.append(t)

    if t > plot_time:
        s = f"Time t = {t:.4f}"
        print(s)

        xdmf.write([u1, w1], t)

        plot_vector(u1, title="Velocity")
        plot_scalar(w1, title="Wave height")

        if time_array:
            plt.figure()
            plt.title("Force")
            plt.plot(time_array, force_array)
            plt.show()

        plot_time += T / plot_freq

    u0.x.array[:] = u1.x.array
    w0.x.array[:] = w1.x.array
    t += dt
    step += 1

xdmf.close()
t_wall = time.time() - t_wall0
print(f"Wall time: {t_wall:.1f} s")

In [ ]:
norm_u1 = float(msh.comm.allreduce(
    assemble_scalar(form(ufl.inner(u1, u1)*ufl.dx)), op=MPI.SUM
))**0.5
norm_w1 = float(msh.comm.allreduce(
    assemble_scalar(form(w1*w1*ufl.dx)), op=MPI.SUM
))**0.5
print(f"||u1|| = {norm_u1:.6f}")
print(f"||w1|| = {norm_w1:.6f}")
print(f"cells  = {n_cells}")
print(f"dt     = {dt:.6f}")
print(f"steps  = {step}")

# **Discussion**

The GLS/SUPG stabilization parameter $\delta_1 = 2/\sqrt{(1/\Delta t)^2 + (|u|/h)^2}$ provides streamline upwinding proportional to the local Courant number. The time-dependent wave-height inflow condition $w = |\sin(t)|$ drives surface waves through the domain past the cylindrical obstacle.